In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
sentences = [
    "我 喜欢 机器 学习",
    "这个 课程 很 有趣",
    "AI 很 强大",
    "学习 深度 学习 很 开心",
    "这个 模型 效果 很 好",

    "我 不 喜欢 这个 课程",
    "这个 模型 效果 很 差",
    "学习 过程 很 痛苦",
    "这个 结果 很 糟糕",
    "我 觉得 很 难"
]

labels = [
    1, 1, 1, 1, 1,
    0, 0, 0, 0, 0
]

In [3]:
all_tokens = []

for sentence in sentences:
    tokens = sentence.split()
    all_tokens.extend(tokens)

unique_tokens = sorted(set(all_tokens))

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for token in unique_tokens:
    vocab[token] = len(vocab)

print(vocab)
print("词表大小:", len(vocab))

{'<PAD>': 0, '<UNK>': 1, 'AI': 2, '不': 3, '喜欢': 4, '好': 5, '学习': 6, '差': 7, '开心': 8, '强大': 9, '很': 10, '我': 11, '效果': 12, '有趣': 13, '机器': 14, '模型': 15, '深度': 16, '痛苦': 17, '糟糕': 18, '结果': 19, '觉得': 20, '课程': 21, '过程': 22, '这个': 23, '难': 24}
词表大小: 25


In [4]:
max_len = 6

encoded_sentences = []

for sentence in sentences:
    tokens = sentence.split()
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]

    while len(ids) < max_len:
        ids.append(vocab["<PAD>"])

    ids = ids[:max_len]
    encoded_sentences.append(ids)

X = torch.tensor(encoded_sentences, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float32).view(-1, 1)

print(X.shape)
print(y.shape)

torch.Size([10, 6])
torch.Size([10, 1])


In [5]:
class SelfAttention(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        self.W_q = nn.Linear(embedding_dim, embedding_dim)
        self.W_k = nn.Linear(embedding_dim, embedding_dim)
        self.W_v = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        d_k = K.shape[-1]

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        attention_weights = F.softmax(scores, dim=-1)

        output = torch.matmul(attention_weights, V)

        return output

In [6]:
class SimpleTransformerBlock(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        self.attention = SelfAttention(embedding_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 2),
            nn.ReLU(),
            nn.Linear(embedding_dim * 2, embedding_dim)
        )

    def forward(self, x):
        attention_output = self.attention(x)

        output = self.feed_forward(attention_output)

        return output

In [7]:
class TransformerTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.transformer_block = SimpleTransformerBlock(embedding_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        embedded = self.embedding(x)

        transformer_output = self.transformer_block(embedded)

        sentence_vector = transformer_output.mean(dim=1)

        output = self.classifier(sentence_vector)

        return output

In [8]:
vocab_size = len(vocab)
embedding_dim = 8

model = TransformerTextClassifier(vocab_size, embedding_dim)

print(model)

TransformerTextClassifier(
  (embedding): Embedding(25, 8, padding_idx=0)
  (transformer_block): SimpleTransformerBlock(
    (attention): SelfAttention(
      (W_q): Linear(in_features=8, out_features=8, bias=True)
      (W_k): Linear(in_features=8, out_features=8, bias=True)
      (W_v): Linear(in_features=8, out_features=8, bias=True)
    )
    (feed_forward): Sequential(
      (0): Linear(in_features=8, out_features=16, bias=True)
      (1): ReLU()
      (2): Linear(in_features=16, out_features=8, bias=True)
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=8, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
    (3): Sigmoid()
  )
)


In [9]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [10]:
for epoch in range(100):
    outputs = model(X)

    loss = criterion(outputs, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"第 {epoch+1} 轮，loss = {loss.item():.4f}")

第 10 轮，loss = 0.5928
第 20 轮，loss = 0.2073
第 30 轮，loss = 0.0026
第 40 轮，loss = 0.0000
第 50 轮，loss = 0.0000
第 60 轮，loss = 0.0000
第 70 轮，loss = 0.0000
第 80 轮，loss = 0.0000
第 90 轮，loss = 0.0000
第 100 轮，loss = 0.0000


In [11]:
def predict_sentiment(sentence):
    tokens = sentence.split()
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]

    while len(ids) < max_len:
        ids.append(vocab["<PAD>"])

    ids = ids[:max_len]

    input_tensor = torch.tensor([ids], dtype=torch.long)

    with torch.no_grad():
        prob = model(input_tensor)
        pred = (prob >= 0.5).float()

    print("输入句子:", sentence)
    print("预测概率:", prob.item())

    if pred.item() == 1:
        print("预测结果: 正面情绪")
    else:
        print("预测结果: 负面情绪")

In [12]:
predict_sentiment("我 喜欢 AI")
predict_sentiment("这个 课程 很 差")
predict_sentiment("学习 很 开心")
predict_sentiment("这个 结果 很 糟糕")

输入句子: 我 喜欢 AI
预测概率: 1.0
预测结果: 正面情绪
输入句子: 这个 课程 很 差
预测概率: 2.2353570817285062e-12
预测结果: 负面情绪
输入句子: 学习 很 开心
预测概率: 1.0
预测结果: 正面情绪
输入句子: 这个 结果 很 糟糕
预测概率: 3.126862310742773e-11
预测结果: 负面情绪
